In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from prophet import Prophet
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Data 
df = pd.read_csv('gas_cleann.csv')
df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')

monthly = df.groupby('date')['value_normalized'].mean().reset_index()
monthly = monthly.sort_values('date').reset_index(drop=True)
monthly = monthly.rename(columns={'date': 'ds', 'value_normalized': 'y'})

prophet = Prophet(yearly_seasonality=True, seasonality_mode='multiplicative')
prophet.fit(monthly)

forecast = prophet.predict(monthly)
monthly['prophet_pred'] = forecast['yhat'].values
monthly['residual'] = monthly['y'] - monthly['prophet_pred']

# CatBoost on Residuals 
monthly['trend'] = np.arange(len(monthly))
monthly['month_sin'] = np.sin(2 * np.pi * monthly['ds'].dt.month / 12)
monthly['month_cos'] = np.cos(2 * np.pi * monthly['ds'].dt.month / 12)

for lag in [1,2,3,6,12]:
    monthly[f'lag_res_{lag}'] = monthly['residual'].shift(lag)

for w in [3,6,12]:
    monthly[f'roll_mean_res_{w}'] = monthly['residual'].rolling(w).mean()

monthly = monthly.dropna().reset_index(drop=True)

features = ['trend', 'month_sin', 'month_cos'] + [f'lag_res_{lag}' for lag in [1,2,3,6,12]] + \
           [f'roll_mean_res_{w}' for w in [3,6,12]]

X = monthly[features]
y_res = monthly['residual']

tscv = TimeSeriesSplit(n_splits=5)

model = CatBoostRegressor(iterations=1500, learning_rate=0.05, depth=6, 
                          l2_leaf_reg=5, early_stopping_rounds=100, 
                          random_seed=42, verbose=False)

r2_list = []

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_res.iloc[train_idx], y_res.iloc[test_idx]
    
    train_pool = Pool(X_train, y_train)
    test_pool  = Pool(X_test, y_test)
    
    model.fit(train_pool, eval_set=test_pool, use_best_model=True)
    
    res_pred = model.predict(X_test)
    final_pred = monthly['prophet_pred'].iloc[test_idx].values + res_pred
    y_true = monthly['y'].iloc[test_idx].values
    
    r2_list.append(r2_score(y_true, final_pred))

print("="*80)
print("Hybrid Prophet + CatBoost")
print(f"R²   : {np.mean(r2_list):.4f} ± {np.std(r2_list):.4f}")
print("="*80)

18:40:33 - cmdstanpy - INFO - Chain [1] start processing
18:40:48 - cmdstanpy - INFO - Chain [1] done processing


Hybrid Prophet + CatBoost
R²   : 0.9612 ± 0.0141
